# SMART CROP DISEASE DETECTION USING DEEP LEARNING (ACRNN)
### B.Tech CSE Major Capstone Project | Department of Computer Science & Engineering
**Team Members:** Ch. Kusuma Priya, D. Pujitha, G. Sasi Charan, G. Revathi  
**Project Guide:** Mrs. N. Rama Devi  
---
This Google Colab notebook provides the GPU-accelerated training pipeline for the **ACRNN (Attentive Convolutional Recurrent Neural Network)** on the PlantVillage crop disease dataset.

## 1. System Environment & GPU Verification

In [ ]:
import tensorflow as tf
import numpy as np
import cv2, os, json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 2. ACRNN Soft Attention Layer

In [ ]:
class ACRNNAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ACRNNAttention, self).__init__(**kwargs)
    def build(self, input_shape):
        dim = input_shape[-1]
        self.W = self.add_weight(name='att_w', shape=(dim, dim), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_b', shape=(dim,), initializer='zeros', trainable=True)
        self.u = self.add_weight(name='att_u', shape=(dim, 1), initializer='glorot_uniform', trainable=True)
        super(ACRNNAttention, self).build(input_shape)
    def call(self, inputs):
        # inputs: (batch, seq_len, dim)
        v = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        scores = tf.tensordot(v, self.u, axes=1)
        weights = tf.nn.softmax(scores, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, weights

## 3. ACRNN Architecture Assembly with MobileNetV2 Backbone

In [ ]:
def build_acrnn(num_classes=27, input_shape=(224, 224, 3)):
    inputs = tf.keras.layers.Input(shape=input_shape, name='leaf_input')
    base_cnn = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=inputs)
    base_cnn.trainable = False
    conv_features = base_cnn.output  # (batch, 7, 7, 1280)
    seq = tf.keras.layers.Reshape((49, 1280), name='spatial_sequence')(conv_features)
    recurrent = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(128, return_sequences=True))(seq)
    context, attn_weights = ACRNNAttention(name='spatial_attention')(recurrent)
    x = tf.keras.layers.Dropout(0.35)(context)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax', name='prediction')(x)
    model = tf.keras.models.Model(inputs=inputs, outputs=[out, attn_weights], name='ACRNN_PlantVillage')
    return model

model = build_acrnn(num_classes=27)
model.summary()

## 4. Dataset Loading & Augmentation

In [ ]:
# Mount Google Drive if dataset is in Drive
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_DIR = 'plantvillage_dataset'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print('Ready to load datasets using tf.keras.utils.image_dataset_from_directory.')

## 5. Compile, Train & Export Weights

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'prediction': 'categorical_crossentropy'},
    metrics={'prediction': ['accuracy']}
)

# After training on dataset:
# model.fit(train_ds, validation_data=val_ds, epochs=15)
# model.save_weights('acrnn_weights.h5')
# print('Saved trained weights to acrnn_weights.h5')